In [ ]:
import numpy as np
import pandas as pd
import re
import ftfy
import html
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
pd.set_option('display.max_colwidth', None)

In [ ]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [ ]:
#TODO:cambiare le print in inglese, in futuro dovrò fare catboost/target encoding
n_nan = df['source'].isna().sum()
n_placeholder = (df['source'] == '\\N').sum()
n_empty = (df['source'].astype(str).str.strip() == '').sum()

print(f"NaN: {n_nan}")
print(f"Placeholder \\N: {n_placeholder}")
print(f"Stringhe vuote: {n_empty}")

df['source'] = df['source'].replace('\\N', np.nan)
df['source'] = df['source'].replace('', np.nan)
df['source'] = df['source'].fillna('Other')

min_freq = 5  
source_counts = df['source'].value_counts()
sources_kept = source_counts[source_counts >= min_freq].index.tolist()

df['source'] = np.where(df['source'].isin(sources_kept),df['source'],'Other')

final_source_counts = df['source'].value_counts()

print(f"Numero categorie finali (incluso Other): {len(final_source_counts)}")

### *Title* feature inspection

In [ ]:
n_nan_title = df['title'].isna().sum()
n_placeholders_title = (df['title']== '\\N').sum()
n_empty_title = (df['title'].astype(str).str.strip()=='').sum()

print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print(f"Number of empty rows: {n_empty_title}")
print("Titles Sample:")
print(df['title'].sample(10))

### *Article* feature inspection

In [ ]:
n_nan_article = df['article'].isna().sum()
n_placeholders_article = (df['article']=='\\N').sum()
n_empty_article = (df['article'].astype(str).str.strip()=='').sum()

print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print(f"Number of empty rows: {n_empty_article}")
print("Articles Sample")
print(df['article'].sample(10))

### *PageRank* feature inspection

In [ ]:
n_nan_pr = df['page_rank'].isna().sum()
n_placeholders_pr = (df['page_rank']=='\\N').sum()
n_empty_pr = (df['page_rank'].astype(str).str.strip()=='').sum()
rank_5=np.array([df['page_rank']==5]).sum()

print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")

### *Timestamp* feature inspection 

In [ ]:
n_nan_time = df['timestamp'].isna().sum()
n_placeholders_time = (df['timestamp']=='\\N').sum()
n_empty_time = (df['timestamp'].astype(str).str.strip()=='').sum()
n_uslesess_time=np.array([df['timestamp']=="0000-00-00 00:00:00"]).sum()

print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

### *Timestamp* feature processing

In [ ]:
def process_timestamp(df):
    df = df.copy()

    dt = pd.to_datetime(df['timestamp'], errors='coerce')

    df['has_date'] = dt.notna().astype(int)
    df['quarter'] = dt.dt.quarter.fillna(-1).astype(int)
    df['is_weekend'] = dt.dt.dayofweek.isin([5, 6]).fillna(False).astype(int)

    df = df.drop(columns=['timestamp'])
    return df

df = process_timestamp(df)

timestamp_cols = ['has_date', 'is_weekend','quarter']
print(f"Timestamp expanded columns: {timestamp_cols}")
print("'timestamp' raw column deleted")
print("Sample of 10 timestamps:")
print(df[timestamp_cols].sample(10))

### *Title* feature stemming

In [ ]:
def clean_text_light(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = ftfy.fix_text(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def process_title_final(df):
    df = df.copy()

    df['title_clean'] = df['title'].apply(clean_text_light)
    df = df.drop(columns=['title'])

    return df


df = process_title_final(df)

print("Sample of 10 cleaned titles:")
print(df['title_clean'].sample(10).to_string(index=False))


### *Article* feature stemming

In [ ]:
##TODO remove comments and change variables, also for title above
def clean_text_light(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = ftfy.fix_text(text)
    text = html.unescape(text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)

    text = re.sub(r'\b(a\s+href|href|img\s+src|nbsp|read\s+more|click\s+here)\b',' ',text,flags=re.IGNORECASE)

    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%\$\€\£]", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def process_article_final(df):
    df = df.copy()

    df['article'] = df['article'].replace('\\N', '').fillna('').astype(str)
    df['article_clean'] = df['article'].apply(clean_text_light)
    df = df.drop(columns=['article'])

    return df

df = process_article_final(df)

print("Sample of 10 cleaned articles:")
print(df['article_clean'].sample(10).to_string(index=False))

In [ ]:
df['text'] = (df['title_clean'] + '[TITLE]' + df['article_clean']).str.strip()
df = df.drop(columns=['title_clean', 'article_clean'])

### Encoding categorical features

In [ ]:
TEXT_COL = "text"
CAT_COLS = ["source"]
NUM_COLS = ["page_rank", "has_date", "is_weekend", "quarter"]
TARGET_COL = "label"

X = df[[TEXT_COL] + CAT_COLS + NUM_COLS].copy()
y = df[TARGET_COL].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.95,
            max_features=150_000,
            sublinear_tf=True  
        ), TEXT_COL),
        ("source_ohe", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=5,
            sparse_output=True
        ), CAT_COLS),
        ("num", "passthrough", NUM_COLS),
    ],
    remainder="drop"
)

svm = LinearSVC(
    class_weight="balanced",
    random_state=42
)

pipe = Pipeline([
    ("prep", preprocess),
    ("clf", svm)
])

param_dist = {
    "clf__C": np.logspace(-3, 1, 10),  
    "clf__loss": ["hinge", "squared_hinge"]
}

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=10,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
print("Best params:", search.best_params_)
print("Best CV Macro F1:", search.best_score_)

y_val_pred = best_model.predict(X_val)
print("Validation Macro F1:", f1_score(y_val, y_val_pred, average="macro"))
print(classification_report(y_val, y_val_pred, digits=4))
